In [1]:
# [1/4] HEALNet 클론 + 패키지 설치
import os, sys

HEALNET_DIR = "/kaggle/working/healnet"
DATASETS    = ["blca", "brca", "kirp", "ucec"]

os.makedirs(HEALNET_DIR, exist_ok=True)

if not os.listdir(HEALNET_DIR):
    !git clone https://github.com/konst-int-i/healnet.git {HEALNET_DIR}

os.chdir(HEALNET_DIR)
print(f"작업 디렉토리: {os.getcwd()}")

!pip install -e . -q
!pip install scikit-survival matplotlib wandb -q
print("설치 완료")

작업 디렉토리: c:\kaggle\working\healnet
설치 완료


In [2]:
# [2/4] gdc-client 설치 (WSI 다운로드용 — 오믹만 쓸 경우 생략 가능)
import os
HEALNET_DIR = "/kaggle/working/healnet"
os.chdir(HEALNET_DIR)

if not os.path.exists(f"{HEALNET_DIR}/gdc-client"):
    !wget -q https://gdc.cancer.gov/files/public/file/gdc-client_v1.6.1_Ubuntu_x64.zip
    !unzip -o gdc-client_v1.6.1_Ubuntu_x64.zip
    !chmod +x gdc-client
    print("gdc-client 설치 완료")
else:
    print("gdc-client 이미 존재")

!./gdc-client --version

gdc-client 이미 존재


'.'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [3]:
# [3/4] git-lfs로 오믹 데이터 받기 (blca / brca / kirp / ucec 포함)
import os
HEALNET_DIR = "/kaggle/working/healnet"
os.chdir(HEALNET_DIR)

!git lfs install
!git lfs pull
!ls data/tcga/omic/

Updated Git hooks.
Git LFS initialized.


'ls'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [4]:
# [4/4] 디렉토리 생성 + best_hyperparams.yml 수정 + loaders.py 패치 (데이터셋 공통 1회 적용)
import os
HEALNET_DIR = "/kaggle/working/healnet"
DATASETS    = ["blca", "brca", "kirp", "ucec"]

# 4개 데이터셋 디렉토리 일괄 생성
for ds in DATASETS:
    os.makedirs(f"{HEALNET_DIR}/data/tcga/wsi/{ds}_preprocessed_level2/patches", exist_ok=True)
    os.makedirs(f"{HEALNET_DIR}/data/tcga/wsi/{ds}", exist_ok=True)
os.makedirs(f"{HEALNET_DIR}/logs", exist_ok=True)
print("디렉토리 생성 완료:", DATASETS)

# ── best_hyperparams.yml 수정 ──────────────────────────────────────────────
hyperparams_path = f"{HEALNET_DIR}/config/best_hyperparams.yml"
with open(hyperparams_path) as f:
    lines = f.readlines()

fix_targets = {"brca", "kirp", "ucec"}
current_ds = None
new_lines = []
for line in lines:
    if not line.startswith(" ") and not line.startswith("#") and ":" in line:
        current_ds = line.split(":")[0].strip()
    if current_ds in fix_targets:
        stripped = line.lstrip()
        indent   = line[: len(line) - len(stripped)]
        if stripped.startswith("num_latents:"):
            line = f"{indent}num_latents: 25\n"
        elif stripped.startswith("latent_dim:"):
            line = f"{indent}latent_dim: 119\n"
    new_lines.append(line)

with open(hyperparams_path, "w") as f:
    f.writelines(new_lines)
print("best_hyperparams.yml 수정 완료")
print("  brca/kirp/ucec: num_latents=25, latent_dim=119 (논문 HEALNet 값으로 교체)")

# ── loaders.py 패치 ───────────────────────────────────────────────────────
os.chdir(HEALNET_DIR)
!git checkout healnet/etl/loaders.py

loaders_path = f"{HEALNET_DIR}/healnet/etl/loaders.py"
with open(loaders_path) as f:
    src = f.read()

# [패치 0] openslide — Colab 등 미설치 환경 호환
src = src.replace(
    "from openslide import OpenSlide",
    "try:\n    from openslide import OpenSlide\nexcept ImportError:\n    OpenSlide = None"
)

with open(loaders_path, "w") as f:
    f.write(src)

# 나머지 5개 패치는 줄 단위로 처리
with open(loaders_path) as f:
    lines = f.readlines()

new_lines = []
i = 0
while i < len(lines):
    line = lines[i]

    # [패치 1] slide_ids — patches 폴더 없을 때 FileNotFoundError 방어
    if "self.slide_ids = [slide_id.rsplit" in line:
        indent = "        "
        new_lines.append(indent + "try:\n")
        new_lines.append(indent + "    " + line.lstrip())
        new_lines.append(indent + "except FileNotFoundError:\n")
        new_lines.append(indent + "    self.slide_ids = []\n")
        i += 1
        continue

    # [패치 2] sample_slide — slide_ids 비어있을 때 IndexError 방어
    if "self.sample_slide_id = self.slide_ids[0]" in line:
        indent = "        "
        new_lines.append(indent + 'if len(self.slide_ids) > 0 and "slides" in self.sources:\n')
        new_lines.append(indent + "    " + line.lstrip())
        new_lines.append(indent + "    " + lines[i + 1].lstrip())
        new_lines.append(indent + "else:\n")
        new_lines.append(indent + "    self.sample_slide_id = None\n")
        new_lines.append(indent + "    self.sample_slide = None\n")
        i += 2
        continue

    # [패치 3] filter_overlap — 슬라이드 없을 때 오믹 샘플 전체가 필터링되는 버그 방지
    if "if self.filter_overlap:" in line:
        new_lines.append(line.replace(
            "if self.filter_overlap:",
            'if self.filter_overlap and "slides" in self.sources:'
        ))
        i += 1
        continue

    # [패치 4] get_resize_dims — sample_slide None일 때 config 치수로 폴백
    if "if override is False:" in line and "sample_slide" not in line:
        new_lines.append(line.replace(
            "if override is False:",
            "if override is False and self.sample_slide is not None:"
        ))
        i += 1
        continue

    # [패치 5] get_info — sample_slide None일 때 슬라이드 정보 출력 건너뜀
    if ('print(f"Slide level count:' in line or
            'print(f"Slide level dimensions:' in line or
            'print(f"Slide resize dimensions:' in line):
        i += 1
        continue

    if 'print(f"Sources selected:' in line:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(indent + "if self.sample_slide is not None:\n")
        new_lines.append(indent + '    print(f"Slide level count: {self.sample_slide.level_count}")\n')
        new_lines.append(indent + '    print(f"Slide level dimensions: {self.sample_slide.level_dimensions}")\n')
        new_lines.append(indent + '    print(f"Slide resize dimensions: w: {self.wsi_width}, h: {self.wsi_height}")\n')

    new_lines.append(line)
    i += 1

with open(loaders_path, "w") as f:
    f.writelines(new_lines)
print("loaders.py 패치 완료 (6개 항목)")

디렉토리 생성 완료: ['blca', 'brca', 'kirp', 'ucec']
best_hyperparams.yml 수정 완료
  brca/kirp/ucec: num_latents=25, latent_dim=119 (논문 HEALNet 값으로 교체)
loaders.py 패치 완료 (6개 항목)


Updated 1 path from the index


In [7]:
# 학습 실행 — 4개 데이터셋 순차 실행
import os
HEALNET_DIR = "/kaggle/working/healnet"
DATASETS    = ["blca", "brca", "kirp", "ucec"]
os.chdir(HEALNET_DIR)

for dataset in DATASETS:
    print(f"\n{'='*50}")
    print(f"  학습 시작: {dataset.upper()}")
    print(f"{'='*50}")

    # 데이터셋별 config 작성
    config_content = f"""data_path: {HEALNET_DIR}/data
tcga_path: {HEALNET_DIR}/data/tcga
gdc_client: {HEALNET_DIR}/gdc-client
log_path: {HEALNET_DIR}/logs
seed: 1
hyperparams: config/best_hyperparams.yml

dataset: {dataset}
model: healnet

explainer: False
missing_ablation: False
omic_attention: True

n_folds: 5

wandb: False

data:
  resize: False
  resize_height: 1024
  resize_width: 1024
  wsi_level: 2
  patch_size: 256

sources:
  - omic

survival:
  loss: nll
  subset: uncensored

train_loop:
  eval_interval: 1
  batch_size: 4
  epochs: 50
  early_stopping: True
  patience: 5

optimizer:
  max_lr: 0.008
  lr: 0.007765016508403882
  momentum: 0.92
  weight_decay: None
"""
    with open(f"{HEALNET_DIR}/config/main_gpu.yml", "w") as f:
        f.write(config_content)

    # 학습 실행 (출력을 화면 + 데이터셋별 로그 파일에 동시 저장)
    log_file = f"/kaggle/working/train_log_{dataset}.txt"
    !WANDB_MODE=disabled python healnet/main.py --config config/main_gpu.yml 2>&1 | tee {log_file}

print("\n모든 데이터셋 학습 완료")


  학습 시작: BLCA

  학습 시작: BRCA

  학습 시작: KIRP

  학습 시작: UCEC

모든 데이터셋 학습 완료


In [6]:
# 결과 요약 — 4개 데이터셋 c-Index 비교
import re, os

DATASETS = ["blca", "brca", "kirp", "ucec"]

# 논문 기준 c-Index — Appendix B Table 3, HEALNet 오믹 단독 결과
paper_baselines = {
    "blca": 0.606,
    "brca": 0.556,
    "kirp": 0.771,
    "ucec": 0.509,
}

results = {}  # {dataset: {fold: score}}

for dataset in DATASETS:
    log_file = f"/kaggle/working/train_log_{dataset}.txt"
    if not os.path.exists(log_file):
        print(f"[{dataset.upper()}] 로그 파일 없음 — 학습 셀을 먼저 실행하세요.")
        continue

    with open(log_file) as f:
        log = f.read()

    sections = re.split(r'\*+FOLD\s+(\d+)\*+', log)
    fold_scores = {}
    for j in range(1, len(sections), 2):
        fold_num = int(sections[j])
        fold_log = sections[j + 1] if j + 1 < len(sections) else ""
        match = re.search(r'test_c_index:\s*([0-9]+\.[0-9]+)', fold_log)
        if match:
            fold_scores[fold_num] = float(match.group(1))
    results[dataset] = fold_scores

# 출력
print("=" * 62)
print("  HEALNet 오믹 단독 실험 결과 (4개 데이터셋)")
print("=" * 62)
print(f"  {'데이터셋':<8}  {'Fold1':>6} {'Fold2':>6} {'Fold3':>6} {'Fold4':>6} {'Fold5':>6}  {'평균':>6}  {'논문':>6}  {'차이':>7}")
print("-" * 62)

for dataset in DATASETS:
    fold_scores = results.get(dataset, {})
    if not fold_scores:
        print(f"  {dataset.upper():<8}  (결과 없음)")
        continue
    avg = sum(fold_scores.values()) / len(fold_scores)
    baseline = paper_baselines.get(dataset)
    fold_str = " ".join(f"{fold_scores.get(k, 0):.4f}" for k in range(1, 6))
    baseline_str = f"{baseline:.3f}" if baseline else "  -   "
    diff_str = f"{avg - baseline:+.4f}" if baseline else "  -   "
    print(f"  {dataset.upper():<8}  {fold_str}  {avg:.4f}  {baseline_str}  {diff_str}")

print("=" * 62)
print("  논문 출처: NeurIPS 2024, Appendix B Table 3 (HEALNet, omic only)")

[BLCA] 로그 파일 없음 — 학습 셀을 먼저 실행하세요.
[BRCA] 로그 파일 없음 — 학습 셀을 먼저 실행하세요.
[KIRP] 로그 파일 없음 — 학습 셀을 먼저 실행하세요.
[UCEC] 로그 파일 없음 — 학습 셀을 먼저 실행하세요.
  HEALNet 오믹 단독 실험 결과 (4개 데이터셋)
  데이터셋       Fold1  Fold2  Fold3  Fold4  Fold5      평균      논문       차이
--------------------------------------------------------------
  BLCA      (결과 없음)
  BRCA      (결과 없음)
  KIRP      (결과 없음)
  UCEC      (결과 없음)
  논문 출처: NeurIPS 2024, Appendix B Table 3 (HEALNet, omic only)
